# Veterinary Clinical Intelligence — Dataset Generation

This notebook generates a synthetic companion animal clinical dataset of ~15,000 records modelled on the structure and prevalence statistics of [VetCompass UK](https://www.rvc.ac.uk/vetcompass) and [SAVSNET](https://www.liverpool.ac.uk/savsnet/).

**Design decisions:**
- Species split: ~53% Dogs (~8,000), ~47% Cats (~7,000) — reflecting VetCompass proportions
- 6 dog breeds, 5 cat breeds — prevalence-weighted
- Diagnosis rates weighted by published VetCompass prevalence data
- Breed-specific predispositions appear at 2–3× the base rate for predisposed breeds
- Age, weight, sex, severity, outcome, and treatment are correlated (not randomly independent)
- ~5–8% of records contain intentional data quality errors for domain-expert cleaning

**Output:** `veterinary_clinical_data.csv` (14 columns)

In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import date, timedelta

np.random.seed(42)
random.seed(42)

TOTAL_RECORDS = 15000
N_DOGS = 8000
N_CATS = 7000

print(f"Generating {TOTAL_RECORDS} records ({N_DOGS} dogs, {N_CATS} cats)...")

## 1. Clinical Reference Data

All weights, age ranges, diagnoses, and prevalence figures are drawn from the `veterinary_clinical_reference.md` document reviewed and approved by Nade (veterinarian, MSc) and Tigar.

In [ ]:
# ── BREED DEFINITIONS ────────────────────────────────────────────────────────

DOG_BREEDS = {
    "Labrador Retriever": {"weight": (25, 36), "age": (1, 14)},
    "French Bulldog":     {"weight": (8, 14),  "age": (1, 10)},
    "German Shepherd":    {"weight": (22, 40), "age": (1, 13)},
    "Yorkshire Terrier":  {"weight": (2, 4),   "age": (1, 16)},
    "Chihuahua":          {"weight": (1.5, 3), "age": (1, 18)},
    "Beagle":             {"weight": (9, 11),  "age": (1, 15)},
}

# Breed popularity weights (reflects real UK registration/practice visit data)
DOG_BREED_WEIGHTS = [0.28, 0.22, 0.18, 0.12, 0.10, 0.10]

CAT_BREEDS = {
    "Domestic Shorthair": {"weight": (3, 6),   "age": (1, 20)},
    "Siamese":            {"weight": (3, 5),   "age": (1, 18)},
    "Persian":            {"weight": (3, 5.5), "age": (1, 17)},
    "Maine Coon":         {"weight": (5, 10),  "age": (1, 15)},
    "British Shorthair":  {"weight": (4, 8),   "age": (1, 17)},
}

CAT_BREED_WEIGHTS = [0.45, 0.15, 0.15, 0.13, 0.12]

# ── DOG GENERAL DISORDERS (VetCompass UK 2021) ───────────────────────────────
# Each entry: (body_system, diagnosis, base_prevalence, presenting_complaint,
#              treatment_type, treatment_detail, severity_options, outcome_options)

DOG_GENERAL = [
    ("Dental",        "Periodontal disease",          0.125,
     "bad breath, difficulty eating",
     "Surgery",       "Dental scaling and extractions",
     ["mild", "moderate"],                 ["recovered"]),
    ("Ear",           "Otitis externa",               0.073,
     "head shaking, ear scratching",
     "Medication",    "Topical antimicrobial/steroid drops, ear flush",
     ["mild", "moderate"],                 ["recovered"]),
    ("GI",            "Diarrhoea / Gastroenteritis",  0.038,
     "diarrhoea, vomiting",
     "Medication",    "Metronidazole, bland diet, fluids",
     ["mild", "moderate"],                 ["recovered"]),
    ("GI",            "Vomiting (acute)",             0.030,
     "vomiting, lethargy",
     "Supportive care", "Anti-emetics (maropitant), fluids, fasting",
     ["mild"],                              ["recovered"]),
    ("GI",            "Foreign body ingestion",       0.012,
     "vomiting, anorexia",
     "Surgery",       "Exploratory laparotomy, endoscopic removal",
     ["severe"],                            ["recovered", "referred"]),
    ("Dermatology",   "Pyoderma",                    0.035,
     "scratching, skin lesions",
     "Medication",    "Systemic antibiotics (cephalexin), topical therapy",
     ["mild", "moderate"],                 ["recovered"]),
    ("Dermatology",   "Atopic dermatitis",            0.020,
     "itching, redness, recurrent skin issues",
     "Medication",    "Oclacitinib (Apoquel), lokivetmab (Cytopoint)",
     ["moderate"],                          ["chronic management"]),
    ("Musculoskeletal", "Osteoarthritis",             0.025,
     "limping, stiffness",
     "Medication",    "NSAIDs (carprofen/meloxicam), joint supplements",
     ["moderate"],                          ["chronic management"]),
    ("Musculoskeletal", "Cruciate ligament rupture",  0.010,
     "acute lameness, non-weight bearing",
     "Surgery",       "TPLO or lateral suture stabilisation",
     ["severe"],                            ["recovered"]),
    ("Urinary",       "Urinary tract infection",      0.015,
     "frequent urination, straining",
     "Medication",    "Antibiotics (amoxicillin-clavulanate)",
     ["mild", "moderate"],                 ["recovered"]),
    ("Respiratory",   "Kennel cough (CIRD)",          0.018,
     "coughing, nasal discharge",
     "Medication",    "Antitussives, antibiotics if secondary infection",
     ["mild"],                              ["recovered"]),
    ("Ophthalmology", "Conjunctivitis",               0.015,
     "eye discharge, redness",
     "Medication",    "Topical ophthalmic antibiotics",
     ["mild"],                              ["recovered"]),
    ("Cardiovascular", "Heart murmur (unspecified)",  0.012,
     "exercise intolerance, coughing",
     "Medication",    "Monitoring, pimobendan if progressive",
     ["moderate"],                          ["chronic management"]),
    ("Reproductive",  "Pyometra",                     0.005,
     "lethargy, vaginal discharge, polydipsia",
     "Surgery",       "Ovariohysterectomy, IV fluids, antibiotics",
     ["severe"],                            ["recovered"]),
]

# ── DOG BREED-SPECIFIC PREDISPOSITIONS ───────────────────────────────────────
# Prevalence here is ADDITIONAL probability for this breed only (base already covered above)

DOG_BREED_SPECIFIC = {
    "Labrador Retriever": [
        ("Musculoskeletal", "Hip dysplasia", 0.060,
         "hindlimb lameness, difficulty rising",
         "Medication", "NSAIDs (carprofen), weight management, joint supplements",
         ["moderate", "severe"], ["chronic management"]),
        ("Ear", "Otitis externa", 0.040,
         "head shaking, ear scratching",
         "Medication", "Ear flushing, topical antimicrobial/steroid drops",
         ["mild", "moderate"], ["recovered"]),
    ],
    "French Bulldog": [
        ("Respiratory", "Brachycephalic Obstructive Airway Syndrome (BOAS)", 0.090,
         "noisy breathing, exercise intolerance, cyanosis",
         "Surgery", "Staphylectomy, nares resection",
         ["moderate", "severe"], ["recovered", "chronic management"]),
        ("Dermatology", "Atopic dermatitis", 0.045,
         "itching, redness, recurrent skin issues",
         "Medication", "Oclacitinib (Apoquel), lokivetmab (Cytopoint), topical therapies",
         ["moderate"], ["chronic management"]),
    ],
    "German Shepherd": [
        ("Musculoskeletal", "Osteoarthritis", 0.050,
         "limping, stiffness",
         "Medication", "NSAIDs (meloxicam), physiotherapy",
         ["moderate"], ["chronic management"]),
        ("Dermatology", "Anal furunculosis", 0.035,
         "scooting, licking perianal area, pain on defecation",
         "Medication", "Ciclosporin, metronidazole",
         ["moderate", "severe"], ["chronic management"]),
        ("Musculoskeletal", "Degenerative myelopathy", 0.020,
         "progressive hindlimb weakness, ataxia",
         "Supportive care", "Physiotherapy, supportive care (no cure)",
         ["severe"], ["chronic management", "deceased"]),
    ],
    "Yorkshire Terrier": [
        ("Respiratory", "Tracheal collapse", 0.060,
         "honking cough, respiratory distress",
         "Medication", "Antitussives (butorphanol), weight management",
         ["mild", "moderate", "severe"], ["chronic management", "recovered"]),
        ("Dental", "Periodontal disease", 0.060,
         "bad breath, difficulty eating",
         "Surgery", "Dental scaling and extractions",
         ["mild", "moderate"], ["recovered"]),
        ("GI", "Portosystemic shunt (PSS)", 0.025,
         "stunted growth, neurological signs, vomiting post-meal",
         "Surgery", "Low-protein diet, lactulose, ameroid constrictor",
         ["severe"], ["recovered", "referred"]),
    ],
    "Chihuahua": [
        ("Cardiovascular", "Myxomatous Mitral Valve Disease (MMVD)", 0.080,
         "exercise intolerance, coughing, syncope",
         "Medication", "Pimobendan, ACE inhibitors, diuretics (furosemide)",
         ["moderate", "severe"], ["chronic management"]),
        ("Musculoskeletal", "Patellar luxation", 0.060,
         "intermittent skipping gait, hindlimb lameness",
         "Surgery", "Trochleoplasty, tibial crest transposition",
         ["mild", "moderate", "severe"], ["recovered"]),
        ("Dental", "Severe dental disease", 0.050,
         "bad breath, difficulty eating, pawing at mouth",
         "Surgery", "Prophylactic dentals, extractions",
         ["mild", "moderate"], ["recovered"]),
    ],
    "Beagle": [
        ("Ear", "Otitis externa", 0.050,
         "head shaking, ear scratching",
         "Medication", "Ear flushing, topical treatment",
         ["mild", "moderate"], ["recovered"]),
        ("GI", "Obesity-related GI issues", 0.040,
         "weight gain, lethargy, reduced exercise tolerance",
         "Supportive care", "Caloric restriction, exercise program",
         ["mild", "moderate"], ["improving", "chronic management"]),
        ("Musculoskeletal", "Intervertebral disc disease (IVDD)", 0.030,
         "back pain, reluctance to jump, hindlimb weakness",
         "Medication", "NSAIDs, cage rest, surgical decompression if severe",
         ["moderate", "severe"], ["recovered", "chronic management"]),
    ],
}

# ── CAT GENERAL DISORDERS (VetCompass UK, n=142,576) ─────────────────────────

CAT_GENERAL = [
    ("Dental",        "Periodontal disease",              0.139,
     "bad breath, drooling, difficulty eating",
     "Surgery",       "Dental scaling and extractions",
     ["mild", "moderate"],                  ["recovered"]),
    ("Dermatology",   "Flea allergy dermatitis",          0.050,
     "scratching, hair loss, skin lesions",
     "Medication",    "Flea treatment (selamectin/fipronil), topical steroids",
     ["mild"],                               ["recovered"]),
    ("GI",            "Vomiting (acute)",                 0.035,
     "vomiting, lethargy",
     "Supportive care", "Anti-emetics (maropitant), fluids",
     ["mild"],                               ["recovered"]),
    ("GI",            "Diarrhoea / Gastroenteritis",      0.025,
     "diarrhoea, decreased appetite",
     "Medication",    "Metronidazole, probiotics, bland diet",
     ["mild", "moderate"],                  ["recovered"]),
    ("Urinary",       "Feline Lower Urinary Tract Disease (FLUTD)", 0.020,
     "straining to urinate, blood in urine, vocalising",
     "Medication",    "Pain relief, urinary diet, fluids; catheterisation if blocked",
     ["moderate", "severe"],                ["recovered", "chronic management"]),
    ("Urinary",       "Chronic Kidney Disease (CKD)",     0.025,
     "weight loss, increased thirst, decreased appetite",
     "Medication",    "Renal diet, phosphate binders, SQ fluids, benazepril",
     ["moderate", "severe"],                ["chronic management"]),
    ("Respiratory",   "Upper respiratory infection (cat flu)", 0.020,
     "sneezing, nasal discharge, eye discharge",
     "Medication",    "Supportive care, antibiotics if secondary (doxycycline)",
     ["mild", "moderate"],                  ["recovered"]),
    ("Musculoskeletal", "Osteoarthritis",                 0.015,
     "reduced jumping, stiffness",
     "Medication",    "Meloxicam (low dose), gabapentin, frunevetmab (Solensia)",
     ["moderate"],                           ["chronic management"]),
    ("Ophthalmology", "Conjunctivitis",                   0.018,
     "eye discharge, squinting",
     "Medication",    "Topical ophthalmic antibiotics",
     ["mild"],                               ["recovered"]),
    ("Cardiovascular", "Hypertrophic cardiomyopathy (HCM)", 0.010,
     "exercise intolerance, open-mouth breathing",
     "Medication",    "Atenolol or diltiazem, clopidogrel",
     ["moderate", "severe"],                ["chronic management"]),
    ("Reproductive",  "Pyometra",                         0.003,
     "lethargy, vaginal discharge, anorexia",
     "Surgery",       "Ovariohysterectomy",
     ["severe"],                             ["recovered"]),
]

# ── CAT BREED-SPECIFIC PREDISPOSITIONS ───────────────────────────────────────

CAT_BREED_SPECIFIC = {
    "Domestic Shorthair": [
        ("Urinary", "Feline Lower Urinary Tract Disease (FLUTD)", 0.030,
         "straining to urinate, blood in urine, vocalising",
         "Medication", "Pain relief, urinary diet, fluids",
         ["moderate", "severe"], ["recovered", "chronic management"]),
        ("Dental", "Periodontal disease", 0.040,
         "bad breath, drooling, difficulty eating",
         "Surgery", "Dental scaling and extractions",
         ["mild", "moderate"], ["recovered"]),
    ],
    "Siamese": [
        ("Respiratory", "Feline asthma", 0.070,
         "wheezing, coughing, laboured breathing",
         "Medication", "Inhaled fluticasone via spacer, oral prednisolone, bronchodilators",
         ["moderate", "severe"], ["chronic management", "recovered"]),
        ("GI", "Mediastinal lymphoma", 0.040,
         "weight loss, dyspnoea, regurgitation",
         "Medication", "Systemic chemotherapy protocols",
         ["severe"], ["chronic management", "referred", "deceased"]),
    ],
    "Persian": [
        ("Urinary", "Polycystic Kidney Disease (PKD)", 0.080,
         "weight loss, vomiting, increased thirst",
         "Medication", "Symptomatic management (fluids, antiemetics, blood pressure control)",
         ["moderate", "severe"], ["chronic management"]),
        ("Ophthalmology", "Corneal ulcer / Sequestrum", 0.050,
         "eye squinting, discharge, corneal pigmentation",
         "Surgery", "Topical ophthalmic antibiotics, lubricants, superficial keratectomy",
         ["moderate", "severe"], ["recovered"]),
    ],
    "Maine Coon": [
        ("Cardiovascular", "Hypertrophic cardiomyopathy (HCM)", 0.090,
         "exercise intolerance, open-mouth breathing, sudden collapse",
         "Medication", "Diuretics (furosemide), ACE inhibitors (benazepril), clopidogrel",
         ["moderate", "severe"], ["chronic management"]),
        ("Musculoskeletal", "Hip dysplasia", 0.040,
         "reduced jumping, hindlimb weakness",
         "Medication", "Weight control, gabapentin or robenacoxib for pain management",
         ["mild", "moderate"], ["chronic management"]),
    ],
    "British Shorthair": [
        ("Cardiovascular", "Hypertrophic cardiomyopathy (HCM)", 0.080,
         "exercise intolerance, open-mouth breathing",
         "Medication", "Beta-blockers (atenolol) or diltiazem, regular echo monitoring",
         ["moderate", "severe"], ["chronic management"]),
        ("GI", "Obesity and Type II Diabetes", 0.050,
         "weight gain, increased thirst, increased urination",
         "Medication", "Strict caloric restriction, high-protein/low-carb diet, insulin glargine",
         ["moderate"], ["chronic management"]),
        ("Cardiovascular", "Feline Aortic Thromboembolism (FATE)", 0.030,
         "sudden hindlimb paralysis, vocalising in pain, cold extremities",
         "Medication", "Analgesia (buprenorphine), antithrombotics (clopidogrel), intensive care",
         ["severe"], ["chronic management", "deceased", "referred"]),
    ],
}

print("Clinical reference data loaded.")

## 2. Helper Functions

In [ ]:
def random_date(start_date, end_date):
    """Return a random date between start_date and end_date."""
    delta = (end_date - start_date).days
    return start_date + timedelta(days=random.randint(0, delta))


def sample_sex(species, diagnosis, age):
    """
    Sample sex with realistic distribution.
    - Reproductive disorders require female (F or FS)
    - Intact females (F) are rare in adult pets (most are spayed/neutered)
    - Older animals more likely to be neutered
    """
    repro_dx = {"Pyometra"}
    if diagnosis in repro_dx:
        return random.choice(["F", "FS"])  # must be female
    # General sex distribution: ~25% MN, ~25% FS, ~25% M, ~25% F
    # Adjust: older animals more likely neutered
    if age >= 5:
        return random.choices(["MN", "FS", "M", "F"], weights=[0.40, 0.40, 0.10, 0.10])[0]
    else:
        return random.choices(["MN", "FS", "M", "F"], weights=[0.25, 0.25, 0.25, 0.25])[0]


def sample_severity(severity_options, age):
    """
    Sample severity — older animals skew toward moderate/severe for chronic conditions.
    """
    if len(severity_options) == 1:
        return severity_options[0]
    # Bias older animals toward higher severity
    if age >= 8 and "severe" in severity_options:
        return random.choices(severity_options,
                              weights=[0.2 if s == "mild" else 0.4 for s in severity_options])[0]
    return random.choice(severity_options)


def sample_outcome(outcome_options, severity):
    """
    Sample outcome — severe cases more likely to result in referral or death.
    """
    if len(outcome_options) == 1:
        return outcome_options[0]
    if severity == "severe" and "referred" in outcome_options:
        weights = [0.5 if o == "referred" else 0.5 / (len(outcome_options) - 1)
                   for o in outcome_options]
        return random.choices(outcome_options, weights=weights)[0]
    return random.choice(outcome_options)


def build_diagnosis_pool(general_disorders, breed_specific, breed):
    """
    Build a weighted pool of disorders combining general prevalence and
    breed-specific predispositions (2–3× boost for predisposed breeds).
    Returns list of (disorder_tuple, weight) pairs.
    """
    pool = []
    # General disorders at base prevalence
    for disorder in general_disorders:
        pool.append((disorder, disorder[2]))  # index 2 = prevalence
    # Breed-specific additions
    if breed in breed_specific:
        for disorder in breed_specific[breed]:
            pool.append((disorder, disorder[2]))
    return pool


print("Helper functions defined.")

## 3. Generate Clean Records

In [ ]:
START_DATE = date(2023, 1, 1)
END_DATE   = date(2024, 12, 31)


def generate_records(n, species, breeds, breed_weights, general_disorders,
                     breed_specific, id_offset=0):
    records = []
    breed_list = list(breeds.keys())

    for i in range(n):
        animal_id = f"ANM{(id_offset + i + 1):05d}"
        visit_id  = f"VIS{(id_offset + i + 1):05d}"

        # Sample breed
        breed = random.choices(breed_list, weights=breed_weights)[0]
        breed_info = breeds[breed]

        # Sample age and weight from breed ranges
        age   = round(random.uniform(*breed_info["age"]), 1)
        weight = round(random.uniform(*breed_info["weight"]), 1)

        # Build diagnosis pool and sample
        pool = build_diagnosis_pool(general_disorders, breed_specific, breed)
        disorders, weights_ = zip(*pool)
        disorder = random.choices(disorders, weights=weights_)[0]

        body_system         = disorder[0]
        diagnosis           = disorder[1]
        presenting_complaint = disorder[3]
        treatment_type      = disorder[4]
        treatment_detail    = disorder[5]
        severity_options    = disorder[6]
        outcome_options     = disorder[7]

        severity = sample_severity(severity_options, age)
        outcome  = sample_outcome(outcome_options, severity)
        sex      = sample_sex(species, diagnosis, age)
        visit_date = random_date(START_DATE, END_DATE).isoformat()

        records.append({
            "animal_id":           animal_id,
            "species":             species,
            "breed":               breed,
            "age_years":           age,
            "weight_kg":           weight,
            "sex":                 sex,
            "visit_id":            visit_id,
            "visit_date":          visit_date,
            "presenting_complaint": presenting_complaint,
            "body_system":         body_system,
            "diagnosis":           diagnosis,
            "severity":            severity,
            "treatment_type":      treatment_type,
            "treatment_detail":    treatment_detail,
            "outcome":             outcome,
        })

    return records


dog_records = generate_records(
    N_DOGS, "Dog",
    DOG_BREEDS, DOG_BREED_WEIGHTS,
    DOG_GENERAL, DOG_BREED_SPECIFIC,
    id_offset=0
)

cat_records = generate_records(
    N_CATS, "Cat",
    CAT_BREEDS, CAT_BREED_WEIGHTS,
    CAT_GENERAL, CAT_BREED_SPECIFIC,
    id_offset=N_DOGS
)

all_records = dog_records + cat_records
random.shuffle(all_records)  # mix species so CSV isn't sorted by species

df = pd.DataFrame(all_records)
print(f"Clean records generated: {len(df):,}")
print(f"Dogs: {(df['species']=='Dog').sum():,}  |  Cats: {(df['species']=='Cat').sum():,}")

## 4. Seed Intentional Data Quality Errors

~5–8% of records will contain one of 9 types of intentional errors, seeded to demonstrate veterinary domain knowledge during the cleaning phase.

| # | Error type | Description |
|---|---|---|
| 1 | Weight outlier | Chihuahua at 25 kg, Maine Coon at 0.5 kg |
| 2 | Age impossibility | Dog >20 years, Cat >25 years |
| 3 | Breed-diagnosis mismatch | BOAS in Labrador (non-brachycephalic) |
| 4 | Treatment-diagnosis mismatch | Insulin prescribed for a fracture |
| 5 | Sex-diagnosis conflict | Pyometra in a male animal |
| 6 | Duplicate breed entry | "Labrador" instead of "Labrador Retriever" |
| 7 | Species-specific impossibility | FLUTD in a dog (feline-only condition) |
| 8 | Missing values | NULL in diagnosis, treatment, or outcome |
| 9 | Date error | Visit date before animal birth or in the future |

In [ ]:
ERROR_RATE = 0.065  # 6.5% of records
n_errors = int(len(df) * ERROR_RATE)

error_indices = random.sample(range(len(df)), n_errors)

# Distribute error types roughly evenly across error indices
error_types = list(range(1, 10))  # 1–9
error_assignments = [error_types[i % len(error_types)] for i in range(n_errors)]
random.shuffle(error_assignments)

df = df.reset_index(drop=True)

for idx, err_type in zip(error_indices, error_assignments):

    if err_type == 1:  # Weight outlier
        breed = df.at[idx, "breed"]
        if breed == "Chihuahua":
            df.at[idx, "weight_kg"] = round(random.uniform(22, 30), 1)
        elif breed == "Maine Coon":
            df.at[idx, "weight_kg"] = round(random.uniform(0.3, 0.8), 1)
        elif df.at[idx, "species"] == "Dog":
            df.at[idx, "weight_kg"] = round(random.uniform(80, 120), 1)
        else:
            df.at[idx, "weight_kg"] = round(random.uniform(0.1, 0.4), 1)

    elif err_type == 2:  # Age impossibility
        if df.at[idx, "species"] == "Dog":
            df.at[idx, "age_years"] = round(random.uniform(21, 30), 1)
        else:
            df.at[idx, "age_years"] = round(random.uniform(26, 35), 1)

    elif err_type == 3:  # Breed-diagnosis mismatch
        if df.at[idx, "species"] == "Dog":
            # BOAS in a non-brachycephalic breed
            non_brachy = ["Labrador Retriever", "German Shepherd", "Beagle"]
            df.at[idx, "breed"] = random.choice(non_brachy)
            df.at[idx, "diagnosis"] = "Brachycephalic Obstructive Airway Syndrome (BOAS)"
            df.at[idx, "body_system"] = "Respiratory"
        else:
            # PKD in a non-Persian cat
            non_persian = ["Domestic Shorthair", "Siamese", "Maine Coon", "British Shorthair"]
            df.at[idx, "breed"] = random.choice(non_persian)
            df.at[idx, "diagnosis"] = "Polycystic Kidney Disease (PKD)"
            df.at[idx, "body_system"] = "Urinary"

    elif err_type == 4:  # Treatment-diagnosis mismatch
        df.at[idx, "diagnosis"] = "Fracture"
        df.at[idx, "body_system"] = "Musculoskeletal"
        df.at[idx, "treatment_detail"] = "Insulin glargine"
        df.at[idx, "treatment_type"] = "Medication"

    elif err_type == 5:  # Sex-diagnosis conflict
        df.at[idx, "diagnosis"] = "Pyometra"
        df.at[idx, "body_system"] = "Reproductive"
        df.at[idx, "sex"] = random.choice(["M", "MN"])  # male with pyometra

    elif err_type == 6:  # Duplicate breed entry (alternate spelling)
        breed_aliases = {
            "Labrador Retriever": "Labrador",
            "German Shepherd": "GSD",
            "Yorkshire Terrier": "Yorkie",
            "Domestic Shorthair": "DSH",
            "British Shorthair": "British SH",
        }
        current_breed = df.at[idx, "breed"]
        if current_breed in breed_aliases:
            df.at[idx, "breed"] = breed_aliases[current_breed]

    elif err_type == 7:  # Species-specific impossibility
        if df.at[idx, "species"] == "Dog":
            # FLUTD is feline-only — assign to a dog
            df.at[idx, "diagnosis"] = "Feline Lower Urinary Tract Disease (FLUTD)"
            df.at[idx, "body_system"] = "Urinary"
        else:
            # Tracheal collapse is a dog condition — assign to a cat
            df.at[idx, "diagnosis"] = "Tracheal collapse"
            df.at[idx, "body_system"] = "Respiratory"

    elif err_type == 8:  # Missing values (NaN)
        col = random.choice(["diagnosis", "treatment_detail", "outcome"])
        df.at[idx, col] = np.nan

    elif err_type == 9:  # Date error
        error_subtype = random.choice(["future", "before_birth"])
        if error_subtype == "future":
            future_date = date(2027, random.randint(1, 12), random.randint(1, 28))
            df.at[idx, "visit_date"] = future_date.isoformat()
        else:
            # Visit before animal could have been born (2020 visit, animal born 2023)
            df.at[idx, "visit_date"] = date(2020, random.randint(1, 12), random.randint(1, 28)).isoformat()
            df.at[idx, "age_years"] = round(random.uniform(3, 6), 1)  # animal 'born' after visit

print(f"Errors seeded: {n_errors} records ({ERROR_RATE*100:.1f}%)")

## 5. Export to CSV

In [ ]:
OUTPUT_PATH = "veterinary_clinical_data.csv"

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

## 6. Sanity Checks

In [ ]:
print("=" * 50)
print("SPECIES SPLIT")
print(df["species"].value_counts())

print("\n" + "=" * 50)
print("BREED DISTRIBUTION")
print(df["breed"].value_counts())

print("\n" + "=" * 50)
print("TOP 10 DIAGNOSES")
print(df["diagnosis"].value_counts().head(10))

print("\n" + "=" * 50)
print("SEVERITY DISTRIBUTION")
print(df["severity"].value_counts())

print("\n" + "=" * 50)
print("OUTCOME DISTRIBUTION")
print(df["outcome"].value_counts(dropna=False))

print("\n" + "=" * 50)
print("MISSING VALUES")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n" + "=" * 50)
print("AGE RANGE (should include some impossibly old animals)")
print(df["age_years"].describe())

print("\n" + "=" * 50)
print("WEIGHT RANGE (should include some outliers)")
print(df["weight_kg"].describe())

print("\n" + "=" * 50)
print("SEEDED ERROR SAMPLES")
print("\n--- Age impossibilities (>20 for dogs, >25 for cats):")
age_errors = df[((df["species"]=="Dog") & (df["age_years"]>20)) |
                ((df["species"]=="Cat") & (df["age_years"]>25))]
print(age_errors[["animal_id","species","breed","age_years"]].head())

print("\n--- Pyometra in males:")
pyo_male = df[(df["diagnosis"]=="Pyometra") & (df["sex"].isin(["M","MN"]))]
print(pyo_male[["animal_id","species","sex","diagnosis"]].head())

print("\n--- FLUTD in dogs (species-specific impossibility):")
flutd_dogs = df[(df["diagnosis"]=="Feline Lower Urinary Tract Disease (FLUTD)") &
                (df["species"]=="Dog")]
print(flutd_dogs[["animal_id","species","diagnosis"]].head())

print("\n--- Future dates:")
future = df[df["visit_date"] > "2025-01-01"]
print(future[["animal_id","visit_date"]].head())

## 7. Preview

In [ ]:
df.head(10)